# Sprint 1 — Pre-processing
**Case Study I — CRM → Billing configuration prediction**

Goal of this sprint: turn the raw `train.csv` / `test.csv` files into a clean, modeling-ready
dataset, and understand the structure of the problem well enough to design Sprint 2 (modeling).

Submission: this notebook, due **22/09/2026 23:59** (40% of the final grade).

Problem recap: given 745 binary CRM configuration columns, predict the 731 binary BILLING
configuration columns (a large multi-label classification problem, evaluated with Exact
Matching Ratio — every label in a row must be correct for that row to count).

## 0. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)

DATA_DIR = '../data'


## 1. Load the data

The raw files are large (train.csv ~560MB, test.csv ~148MB). Everything except `MSISDN` is
binary (0/1), so we load as `int8` / `uint8` to keep memory manageable — this alone cuts
memory use by roughly 8x vs. the pandas default `int64`.

In [2]:
# Peek at the header first to build a dtype map without loading everything as int64 first
header = pd.read_csv(f'{DATA_DIR}/train.csv', nrows=0).columns.tolist()
crm_cols = [c for c in header if c.startswith('CRM')]
bil_cols = [c for c in header if c.startswith('BIL')]
print(f'{len(crm_cols)} CRM columns, {len(bil_cols)} BIL columns, {len(header)} total')

dtype_map = {c: 'int8' for c in crm_cols + bil_cols}
dtype_map['MSISDN'] = str

train = pd.read_csv(f'{DATA_DIR}/train.csv', dtype=dtype_map)
print('train shape:', train.shape)
train.head()

745 CRM columns, 731 BIL columns, 1477 total
train shape: (187442, 1477)


,MSISDN,CRM_BUSINESS_LINE_ORIG_Mobile,CRM_1519_PACK,CRM_1521_PACK,CRM_1522_PACK,CRM_1523_PACK,CRM_1524_PACK,CRM_1525_PACK,CRM_1526_PACK,CRM_1527_PACK,CRM_1528_PACK,CRM_1529_PACK,CRM_1530_PACK,CRM_1532_PACK,CRM_1533_PACK,CRM_1534_PACK,CRM_1536_PACK,CRM_1537_PACK,CRM_1538_PACK,CRM_1539_PACK,CRM_1540_PACK,CRM_1541_PACK,CRM_1542_PACK,CRM_1543_PACK,CRM_1545_PACK,...,BIL_4449_PACK,BIL_4450_PACK,BIL_4451_PACK,BIL_4452_PACK,BIL_4454_PACK,BIL_4455_PACK,BIL_4456_PACK,BIL_4457_PACK,BIL_4458_PACK,BIL_4459_PACK,BIL_4460_PACK,BIL_4461_PACK,BIL_4462_PACK,BIL_4463_PACK,BIL_4464_PACK,BIL_4465_PACK,BIL_4466_PACK,BIL_4467_PACK,BIL_4468_PACK,BIL_4504_PACK,BIL_4506_PACK,BIL_4507_PACK,BIL_4508_PACK,BIL_4509_PACK,BIL_4510_PACK
0,4f1b9e575ac7ea93b4a978b3095d7ee1,1,0,0,0,0,0,0,0,0,1,1,0,1,1,0,1,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,01908281632545a2fc22d4a6365f2b27,1,0,0,0,0,0,0,0,0,0,1,0,1,1,1,1,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,f6f0c782716d8928928407c4e426be25,1,0,0,0,0,0,0,0,0,0,1,0,1,1,0,1,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,a543952f2405c440fc40f85e8adfdfd6,1,0,0,0,0,0,0,0,0,0,1,0,1,1,0,1,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,63e3cdf2eff067ce9af9f9f8530b16eb,1,0,0,0,0,0,0,0,0,0,1,0,1,1,0,1,0,1,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [3]:
test_dtype_map = {c: 'int8' for c in crm_cols}
test_dtype_map['MSISDN'] = str
test = pd.read_csv(f'{DATA_DIR}/test.csv', dtype=test_dtype_map)
print('test shape:', test.shape)
test.head()

test shape: (97100, 746)


,MSISDN,CRM_BUSINESS_LINE_ORIG_Mobile,CRM_1519_PACK,CRM_1521_PACK,CRM_1522_PACK,CRM_1523_PACK,CRM_1524_PACK,CRM_1525_PACK,CRM_1526_PACK,CRM_1527_PACK,CRM_1528_PACK,CRM_1529_PACK,CRM_1530_PACK,CRM_1532_PACK,CRM_1533_PACK,CRM_1534_PACK,CRM_1536_PACK,CRM_1537_PACK,CRM_1538_PACK,CRM_1539_PACK,CRM_1540_PACK,CRM_1541_PACK,CRM_1542_PACK,CRM_1543_PACK,CRM_1545_PACK,...,CRM_2370_PACK,CRM_2371_PACK,CRM_2372_PACK,CRM_2373_PACK,CRM_2374_PACK,CRM_2375_PACK,CRM_2376_PACK,CRM_2377_PACK,CRM_2378_PACK,CRM_2379_PACK,CRM_2380_PACK,CRM_2381_PACK,CRM_2382_PACK,CRM_2383_PACK,CRM_2385_PACK,CRM_2387_PACK,CRM_2388_PACK,CRM_2389_PACK,CRM_2391_PACK,CRM_2392_PACK,CRM_2394_PACK,CRM_2395_PACK,CRM_2396_PACK,CRM_2397_PACK,CRM_2398_PACK
0,db67b03cf9fb8ea96077cd34c7a28139,1,0,1,1,1,0,0,1,1,1,1,0,1,1,0,1,0,0,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2ac0d310d5cf1a96587af614d37b8b54,1,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0,1,1,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,a8f289fbd263794ac74059a18c1d10a5,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,5a53a1c85ccc7e794c90c75b75b8d3cb,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,484b400226b8894377e12204219283fc,1,0,0,0,0,0,0,0,0,1,1,0,1,1,0,1,0,0,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 2. Basic sanity checks

- Any missing values?
- Any duplicate MSISDN?
- How sparse are the CRM / BIL configurations (mean activation rate per column)?

In [4]:
print('Missing values in train:', train.isna().sum().sum())
print('Missing values in test:', test.isna().sum().sum())
print('Duplicate MSISDN in train:', train['MSISDN'].duplicated().sum())
print('Duplicate MSISDN in test:', test['MSISDN'].duplicated().sum())

print('\nMean activation rate, CRM columns:', train[crm_cols].mean().mean().round(4))
print('Mean activation rate, BIL columns:', train[bil_cols].mean().mean().round(4))

Missing values in train: 0
Missing values in test: 0
Duplicate MSISDN in train: 0
Duplicate MSISDN in test: 0

Mean activation rate, CRM columns: 0.0205
Mean activation rate, BIL columns: 0.0201


In [5]:
crm_var = train[crm_cols].nunique()
bil_var = train[bil_cols].nunique()

constant_crm = crm_var[crm_var <= 1].index.tolist()
constant_bil = bil_var[bil_var <= 1].index.tolist()

print(f'Constant (zero-variance) CRM columns: {len(constant_crm)} / {len(crm_cols)}')
print(f'Constant (zero-variance) BIL columns: {len(constant_bil)} / {len(bil_cols)}')

Constant (zero-variance) CRM columns: 2 / 745
Constant (zero-variance) BIL columns: 0 / 731


## 3. The CRM → BILLING relationship

The brief says the mapping can be many-to-one (different CRM configs → same BILLING config),
**and** that the training data contains provisioning errors — i.e. for some identical CRM
configs, different BILLING configs appear. Let's measure this directly, since it shapes how
noisy Sprint 2's target really is.

In [6]:
# Encode each row's CRM config and BIL config as a single string key so we can group on it
crm_key = train[crm_cols].astype(str).agg(''.join, axis=1)
bil_key = train[bil_cols].astype(str).agg(''.join, axis=1)

n_unique_crm = crm_key.nunique()
n_unique_bil = bil_key.nunique()
print(f'Unique CRM configs: {n_unique_crm} ({n_unique_crm/len(train):.1%} of rows)')
print(f'Unique BIL configs: {n_unique_bil} ({n_unique_bil/len(train):.1%} of rows)')

pairs = pd.DataFrame({'crm': crm_key, 'bil': bil_key})
ambiguous = pairs.groupby('crm')['bil'].nunique()
n_ambiguous = (ambiguous > 1).sum()
print(f'CRM configs mapping to >1 distinct BIL config (likely provisioning errors): '
      f'{n_ambiguous} / {len(ambiguous)} unique CRM configs '
      f'({n_ambiguous/len(ambiguous):.1%})')

Unique CRM configs: 67434 (36.0% of rows)
Unique BIL configs: 67201 (35.9% of rows)
CRM configs mapping to >1 distinct BIL config (likely provisioning errors): 617 / 67434 unique CRM configs (0.9%)
